<a href="https://colab.research.google.com/github/TASRUNNER2010/Hackathon/blob/main/Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from pathlib import Path

# Locate the folder containing all four original EPA files.
folder = Path(
    "/content/drive/MyDrive/veritas AI project"
)

# List only the original files so we don't accidentally load cleaned outputs.
filenames = [
    "2023_ad_viz_plotval_data.csv",
    "2024_ad_viz_plotval_data.csv",
    "2025_ad_viz_plotval_data.csv",
    "2026_ad_viz_plotval_data.csv"
]

# Read each file and preserve identifiers as text.
tables = []

for filename in filenames:
    table = pd.read_csv(
        folder / filename,
        dtype={"Site ID": "string", "POC": "string"}
    )

    # Remove surrounding spaces from column names.
    table.columns = table.columns.str.strip()

    # Record which file each measurement came from.
    table["source_file"] = filename
    tables.append(table)

# Combine the years and preserve an independent raw copy.
raw_df = pd.concat(tables, ignore_index=True)

# Select the Rutgers station and its mostly daily POC 3 series.
df = raw_df.loc[
    raw_df["Site ID"].str.strip().eq("340230011")
    & raw_df["POC"].str.strip().eq("3")
].copy()

# Stop if the expected station and instrument were not found.
if df.empty:
    raise ValueError("No records found for Rutgers POC 3.")

# Keep measurements and metadata useful for checking data quality.
df = df[[
    "Date",
    "Daily Mean PM2.5 Concentration",
    "Daily AQI Value",
    "Source",
    "Method Code",
    "Method Description",
    "source_file"
]].rename(columns={
    "Date": "date",
    "Daily Mean PM2.5 Concentration": "pm25",
    "Daily AQI Value": "aqi",
    "Source": "source",
    "Method Code": "method_code",
    "Method Description": "method_description"
})

# Convert date strings into dates and measurements into numbers.
df["date"] = pd.to_datetime(
    df["date"], format="%m/%d/%Y", errors="raise"
)

for column in ["pm25", "aqi"]:
    df[column] = pd.to_numeric(df[column], errors="raise")

# Remove repeated records, ignoring the filename when comparing rows.
record_columns = [c for c in df.columns if c != "source_file"]
df = df.drop_duplicates(subset=record_columns)

# Investigate conflicting same-day records rather than choosing arbitrarily.
if df["date"].duplicated().any():
    raise ValueError("Multiple different records exist for the same date.")

# Build a complete calendar from the actual selected date range.
# This replaces the old calendar that was restricted to 2023.
calendar = pd.date_range(
    start=df["date"].min(),
    end=df["date"].max(),
    name="date"
)

# Insert missing dates without inventing pollution measurements.
df = (
    df.set_index("date")
      .sort_index()
      .reindex(calendar)
      .reset_index()
)

# Flag missing pollution measurements and preliminary AirNow records.
df["measurement_missing"] = df["pm25"].isna()
df["is_airnow"] = df["source"].eq("AirNow")

# Summarize measurement coverage separately for each year.
summary = df.groupby(df["date"].dt.year).agg(
    calendar_days=("date", "size"),
    measured_days=("pm25", "count"),
    missing_days=("measurement_missing", "sum"),
    airnow_records=("is_airnow", "sum")
)

display(summary)
display(df.head())

# Save a new combined file without overwriting any original CSV.
df.to_csv(folder / "airaware_epa_clean_daily_2023_2026.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/veritas AI project/2023_ad_viz_plotval_data.csv'

In [ ]:
import requests

# Use the first and last dates in the cleaned EPA dataset.
# strftime converts dates into the format Open-Meteo expects.
start_date = df["date"].min().strftime("%Y-%m-%d")
end_date = df["date"].max().strftime("%Y-%m-%d")

# Open-Meteo's historical weather API.
url = "https://archive-api.open-meteo.com/v1/archive"

# Specify the location, date range, weather variables, and units.
params = {
    # Coordinates of the Rutgers EPA monitoring station.
    "latitude": 40.462182,
    "longitude": -74.429439,

    # Match the full date range of the cleaned EPA data.
    "start_date": start_date,
    "end_date": end_date,

    # Daily average temperature, humidity, and wind speed,
    # plus total daily precipitation.
    "daily": ",".join([
        "temperature_2m_mean",
        "relative_humidity_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_mean"
    ]),

    # Match weather days to local New Jersey calendar days.
    "timezone": "America/New_York",

    # Request explicit measurement units.
    "temperature_unit": "celsius",
    "precipitation_unit": "mm",
    "wind_speed_unit": "ms"
}

# Download the weather data.
# Stop if Open-Meteo returns an HTTP error.
response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

# Convert the response from JSON into a daily weather table.
weather_data = response.json()
weather = pd.DataFrame(weather_data["daily"])

# Shorten column names and include units in their names.
weather = weather.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature_c",
    "relative_humidity_2m_mean": "humidity_pct",
    "precipitation_sum": "precipitation_mm",
    "wind_speed_10m_mean": "wind_speed_ms"
})

# Convert weather date strings into pandas dates.
weather["date"] = pd.to_datetime(weather["date"])

# Ensure weather values are numeric.
weather_columns = [
    "temperature_c",
    "humidity_pct",
    "precipitation_mm",
    "wind_speed_ms"
]

for column in weather_columns:
    weather[column] = pd.to_numeric(weather[column], errors="raise")

# Match weather and pollution records by date.
# Preserve every EPA calendar day, including missing measurements.
# Stop if either table has duplicate dates.
combined = df.merge(
    weather,
    on="date",
    how="left",
    validate="one_to_one"
).sort_values("date").reset_index(drop=True)

# Report the requested period and missing weather values.
print(f"Requested weather: {start_date} to {end_date}")
print("Missing weather values:")
print(combined[weather_columns].isna().sum())

# Show a sample of the merged pollution-and-weather dataset.
display(combined.head(10))

# Save weather separately so it can be reused without another download.
weather.to_csv(
    folder / "airaware_weather_2023_2026.csv",
    index=False
)

# Save the complete merged dataset without overwriting the EPA-only file.
combined.to_csv(
    folder / "airaware_epa_weather_2023_2026.csv",
    index=False
)

In [ ]:
# Work on a separate copy of the complete daily calendar.
forecast_data = combined.sort_values("date").copy()

# Use the following calendar day's PM2.5 as the prediction target.
forecast_data["pm25_tomorrow"] = forecast_data["pm25"].shift(-1)

# Record the date being predicted so we can split by target date.
forecast_data["target_date"] = (
    forecast_data["date"] + pd.Timedelta(days=1)
)

# Start with today's pollution and today's weather as inputs.
features = [
    "pm25",
    "temperature_c",
    "humidity_pct",
    "precipitation_mm",
    "wind_speed_ms"
]

# Keep only examples with complete inputs and a real next-day measurement.
# Missing pollution is never replaced with zero or an invented target.
model_df = forecast_data.dropna(
    subset=features + ["pm25_tomorrow"]
).copy()

# Split chronologically using the date being predicted.
train_df = model_df[
    model_df["target_date"] < "2025-01-01"
].copy()

validation_df = model_df[
    (model_df["target_date"] >= "2025-01-01")
    & (model_df["target_date"] < "2026-01-01")
].copy()

test_df = model_df[
    model_df["target_date"] >= "2026-01-01"
].copy()

# Check how many forecasting examples are available in each period.
print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Test examples:", len(test_df))

# Inspect the input date, target date, and actual next-day pollution.
display(model_df[
    ["date", "target_date", "pm25", "pm25_tomorrow"]
].head())

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Select the training inputs and the next-day measurements to learn from.
X_train = train_df[features]
y_train = train_df["pm25_tomorrow"]

# Create a Linear Regression model.
model = LinearRegression()

# Train the model using only the training period.
model.fit(X_train, y_train)

# Predict next-day pollution for the validation examples.
validation_predictions = model.predict(validation_df[features])

# Baseline: tomorrow's pollution equals today's pollution.
baseline_predictions = validation_df["pm25"]

# Get the actual next-day measurements for those same examples.
actual = validation_df["pm25_tomorrow"]

# Compare both approaches using mean absolute error.
baseline_mae = mean_absolute_error(actual, baseline_predictions)
model_mae = mean_absolute_error(actual, validation_predictions)

# Lower MAE means smaller prediction errors on average.
print(f"Validation baseline MAE: {baseline_mae:.2f} µg/m³")
print(f"Validation Linear Regression MAE: {model_mae:.2f} µg/m³")

In [ ]:
# Predicted pollution levels vs Actual (Jan 2023 - Jan 2025)
import matplotlib.pyplot as plt

# Create a graph comparing actual and predicted PM2.5.
plt.figure(figsize=(14, 6))

plt.plot(
    validation_df["target_date"],
    actual.values,
    label="Actual PM2.5",
    linewidth=2
)

plt.plot(
    validation_df["target_date"],
    validation_predictions,
    label="Predicted PM2.5",
    linewidth=2,
    linestyle="--"
)

# Add the project warning threshold.
plt.axhline(
    y=35,
    color="red",
    linestyle=":",
    label="35 µg/m³ Warning Threshold"
)

plt.title("Actual vs. Predicted Next-Day PM2.5", fontsize=16)
plt.xlabel("Date")
plt.ylabel("PM2.5 Concentration (µg/m³)")

plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Test graph (Not final): Baseline vs Predicted
# Compare the Mean Absolute Errors.
models = ["Baseline", "Linear Regression"]
errors = [baseline_mae, model_mae]

plt.figure(figsize=(8, 5))

bars = plt.bar(models, errors)

plt.title("Prediction Error Comparison", fontsize=16)
plt.ylabel("Mean Absolute Error (µg/m³)")
plt.xlabel("Prediction Model")

# Display the actual MAE above each bar.
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.2f}",
        ha="center",
        va="bottom"
    )

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()


In [ ]:
# Test graph (Not final)
plt.figure(figsize=(7, 7))

plt.scatter(
    actual,
    validation_predictions,
    alpha=0.6,
    label="Predictions"
)

# Perfect prediction reference line.
minimum = min(actual.min(), validation_predictions.min())
maximum = max(actual.max(), validation_predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--",
    color="red",
    label="Perfect Predictions"
)

plt.title("Actual vs. Predicted PM2.5")
plt.xlabel("Actual PM2.5 (µg/m³)")
plt.ylabel("Predicted PM2.5 (µg/m³)")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# Install the library used to create the interface in Colab.
!pip -q install gradio

In [ ]:
import gradio as gr
import pandas as pd
import math

# Receive inputs from the interface and predict next-day pollution.
def predict_pm25(pm25, temperature, humidity, precipitation, wind):
    values = [pm25, temperature, humidity, precipitation, wind]

    # Require a finite number in every input field.
    if any(v is None or not math.isfinite(float(v)) for v in values):
        raise gr.Error("Please enter a valid number in every field.")

    # Check physical ranges before passing inputs to the model.
    if pm25 < 0 or precipitation < 0 or wind < 0:
        raise gr.Error("PM2.5, precipitation, and wind cannot be negative.")

    if not 0 <= humidity <= 100:
        raise gr.Error("Humidity must be between 0 and 100 percent.")

    # Match the input column names and order used during training.
    inputs = pd.DataFrame([{
        "pm25": pm25,
        "temperature_c": temperature,
        "humidity_pct": humidity,
        "precipitation_mm": precipitation,
        "wind_speed_ms": wind
    }])[features]

    # Predict using the already-trained Linear Regression model.
    prediction = float(model.predict(inputs)[0])

    # Persistence baseline: tomorrow equals today.
    baseline = float(pm25)

    # Explain negative results without silently changing predictions.
    if prediction < 0:
        interpretation = (
            "The model returned a physically impossible negative value. "
            "This is a limitation of unconstrained Linear Regression."
        )
    elif prediction > 35:
        interpretation = (
            "Prediction exceeds the illustrative project threshold "
            "of 35 µg/m³. This is not an official air-quality advisory."
        )
    else:
        interpretation = (
            "Prediction does not exceed the illustrative project threshold "
            "of 35 µg/m³. This does not establish that outdoor activity is safe."
        )

    return (
        f"{prediction:.2f} µg/m³",
        f"{baseline:.2f} µg/m³",
        interpretation
    )


# Supply clickable historical inputs from the validation period.
# These examples do not use the reserved 2026 test set.
examples = validation_df[features].head(5).to_numpy().tolist()

# Build the browser form.
demo = gr.Interface(
    fn=predict_pm25,

    inputs=[
        gr.Number(label="Input day's average PM2.5 (µg/m³)", minimum=0),
        gr.Number(label="Input day's average temperature (°C)"),
        gr.Number(
            label="Input day's average humidity (%)",
            minimum=0,
            maximum=100
        ),
        gr.Number(label="Input day's total precipitation (mm)", minimum=0),
        gr.Number(label="Input day's average wind speed (m/s)", minimum=0)
    ],

    outputs=[
        gr.Textbox(label="Predicted next-day PM2.5"),
        gr.Textbox(label="Baseline: tomorrow equals input day"),
        gr.Textbox(label="Interpretation")
    ],

    examples=examples,

    title="AirAware — Historical Forecasting Prototype",

    description=(
        "Estimate next-day average PM2.5 near the Rutgers monitoring "
        "station using a completed day's pollution and weather. "
        "Click a historical example below, then click Predict. "
        "This prototype does not retrieve live measurements or forecasts."
    ),

    submit_btn="Predict",
    clear_btn="Clear",

    # Calculate example predictions when clicked, using the current model.
    cache_examples=False
)

# Display the interface in Colab.
# The Colab runtime must remain running to use it.
demo.launch(share=False)

In [ ]:
# ── AirAware export cell ─────────────────────────────────────────────
# Paste this as a NEW cell at the end of Hackathon.ipynb and run it after
# the training cell. It writes airaware_export.json, the one file the
# AirAware website reads. Every number on the site comes from here:
# the site never recomputes a forecast.
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

THRESHOLD = 35.0          # experimental project threshold, µg/m³
MODEL_VERSION = "1.0.0"   # bump this when you retrain


def _num(value, digits=2):
    """Round a number for JSON; keep missing values as null."""
    return None if pd.isna(value) else round(float(value), digits)


def _day(value):
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def _col(frame, name, digits=2):
    return [_num(v, digits) for v in frame[name]]


# 1. Every model is a function from a table of examples to predictions.
#    Register a new model by adding one line here.
#    The weather heuristic is a fixed rule of thumb, not a trained model.
#    Its wind cut-offs come from the training period only. The multipliers
#    are a starting guess: change them and re-run to test your own rule.
CALM_WIND = float(train_df["wind_speed_ms"].quantile(0.25))
BREEZY_WIND = float(train_df["wind_speed_ms"].quantile(0.75))


def weather_heuristic(frame):
    wind = frame["wind_speed_ms"].to_numpy(dtype=float)
    rain = frame["precipitation_mm"].to_numpy(dtype=float)
    factor = np.ones(len(frame))
    factor[wind >= BREEZY_WIND] *= 0.85          # windy day: expect cleaner air
    factor[(wind <= CALM_WIND) & (rain < 1)] *= 1.10   # still, dry day: build-up
    factor[rain >= 1] *= 0.90                    # rain washes particles out
    return frame["pm25"].to_numpy(dtype=float) * factor


model_outputs = {
    "persistence": lambda frame: frame["pm25"].to_numpy(dtype=float),
    "weather_heuristic": weather_heuristic,
    "linear_regression": lambda frame: model.predict(frame[features]),
}

# 2. Predictions and MAE for each evaluation period (same rows for all models).
eval_sets = {"validation": validation_df, "test": test_df}
predictions = {}
mae = {name: {} for name in model_outputs}

for split, frame in eval_sets.items():
    block = {
        "date": [_day(d) for d in frame["date"]],
        "target_date": [_day(d) for d in frame["target_date"]],
        "actual": _col(frame, "pm25_tomorrow"),
    }
    for name, predict in model_outputs.items():
        values = predict(frame)
        block[name] = [round(float(v), 2) for v in values]
        mae[name][split] = round(
            float(mean_absolute_error(frame["pm25_tomorrow"], values)), 3
        )
    predictions[split] = block

# 3. Tomorrow's forecast from the most recent day with complete inputs.
latest_frame = forecast_data.dropna(subset=features).iloc[[-1]]
latest = latest_frame.iloc[0]
train_mean = train_df[features].mean()
train_std = train_df[features].std()

forecast = {
    "input_date": _day(latest["date"]),
    "target_date": _day(latest["date"] + pd.Timedelta(days=1)),
    "units": "µg/m³",
    "values": {
        name: _num(predict(latest_frame)[0])
        for name, predict in model_outputs.items()
    },
    # How far each input pushes the Linear Regression forecast away from
    # the training-period average prediction.
    "explanation": {
        "base": _num(model.intercept_ + float(np.dot(model.coef_, train_mean))),
        "contributions": {
            f: _num(c * (float(latest[f]) - train_mean[f]), 3)
            for f, c in zip(features, model.coef_)
        },
    },
}


def _period(frame):
    return {
        "start": _day(frame["target_date"].min()),
        "end": _day(frame["target_date"].max()),
        "examples": int(len(frame)),
    }


# 4. Model registry.
models = [
    {
        "id": "persistence",
        "name": "Persistence baseline",
        "version": MODEL_VERSION,
        "kind": "baseline",
        "features": ["pm25"],
        "training_period": None,
        "mae": mae["persistence"],
        "explanation": "Assumes tomorrow's PM2.5 equals today's. No training. "
                       "Any real model has to beat this.",
    },
    {
        "id": "weather_heuristic",
        "name": "Weather heuristic",
        "version": MODEL_VERSION,
        "kind": "rule of thumb",
        "features": ["pm25", "wind_speed_ms", "precipitation_mm"],
        "training_period": None,
        "mae": mae["weather_heuristic"],
        "explanation": "Starts from today's PM2.5, then applies fixed rules: "
                       f"15% lower after a windy day (wind at or above {BREEZY_WIND:.1f} m/s), "
                       f"10% higher after a still, dry day (wind at or below {CALM_WIND:.1f} m/s), "
                       "10% lower after 1 mm or more of rain. Nothing is fitted; "
                       "only the two wind cut-offs come from the training period.",
    },
    {
        "id": "linear_regression",
        "name": "Linear Regression",
        "version": MODEL_VERSION,
        "kind": "scikit-learn LinearRegression",
        "features": list(features),
        "training_period": _period(train_df),
        "mae": mae["linear_regression"],
        "intercept": _num(model.intercept_, 8),
        "coefficients": {f: _num(c, 8) for f, c in zip(features, model.coef_)},
        "feature_mean": {f: _num(train_mean[f], 3) for f in features},
        "feature_std": {f: _num(train_std[f], 3) for f in features},
        "explanation": "Starts from today's PM2.5 and adjusts it using today's "
                       "temperature, humidity, rain and wind, with weights "
                       "learned from the training period only.",
    },
]

# 5. Dataset coverage, straight from the cleaning step.
years = combined["date"].dt.year
coverage = [
    {
        "year": int(year),
        "calendar_days": int(len(group)),
        "measured_days": int(group["pm25"].notna().sum()),
        "missing_days": int(group["pm25"].isna().sum()),
        "airnow_records": int(group["source"].eq("AirNow").sum()),
    }
    for year, group in combined.groupby(years)
]

export = {
    "schema_version": "1",
    "demo": False,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "units": "µg/m³",
    "threshold": {
        "value": THRESHOLD,
        "label": "Experimental project threshold",
    },
    "location": {
        "id": "340230011-3",
        "name": "Rutgers University",
        "county": "Middlesex County",
        "state": "NJ",
        "site_id": "340230011",
        "poc": "3",
        "latitude": 40.462182,
        "longitude": -74.429439,
        "method": "Thermo Scientific 5014i or FH62C14-DHS w/VSCC (method 183)",
    },
    "sources": {
        "pollution": "US EPA daily PM2.5 (AQS, with preliminary AirNow records)",
        "weather": "Open-Meteo historical weather archive",
    },
    "daily": {
        "date": [_day(d) for d in combined["date"]],
        "pm25": _col(combined, "pm25", 1),
        "aqi": _col(combined, "aqi", 0),
        "source": [None if pd.isna(s) else str(s) for s in combined["source"]],
        "temperature_c": _col(combined, "temperature_c", 1),
        "humidity_pct": _col(combined, "humidity_pct", 0),
        "precipitation_mm": _col(combined, "precipitation_mm", 1),
        "wind_speed_ms": _col(combined, "wind_speed_ms", 2),
    },
    "splits": {
        "train": _period(train_df),
        "validation": _period(validation_df),
        "test": _period(test_df),
    },
    "models": models,
    "predictions": predictions,
    "forecast": forecast,
    "coverage": coverage,
}

out_path = folder / "airaware_export.json"
with open(out_path, "w", encoding="utf-8") as fh:
    json.dump(export, fh, ensure_ascii=False, separators=(",", ":"))

print("Wrote", out_path)
print("Forecast for", forecast["target_date"], "->", forecast["values"])
print("MAE:", mae)


In [ ]:
# ── AirAware viewer cell ─────────────────────────────────────────────
# Run this AFTER the export cell. It shows the AirAware site inside the
# notebook, filled with the forecast your model just produced.
# Needs index.html uploaded to the same Drive folder as your CSV files.
import html
import json
from IPython.display import HTML, display

# Let the page's "Predict a day" form ask the trained model directly.
try:
    from google.colab import output
    from IPython.display import JSON
    import pandas as pd

    def _airaware_predict(pm25, temperature, humidity, precipitation, wind):
        X = pd.DataFrame([{
            "pm25": pm25, "temperature_c": temperature, "humidity_pct": humidity,
            "precipitation_mm": precipitation, "wind_speed_ms": wind,
        }])[features].astype(float)
        return JSON({name: float(predict(X)[0]) for name, predict in model_outputs.items()})

    output.register_callback("airaware.predict", _airaware_predict)
except ImportError:
    pass  # not running in Colab: the page applies the exported model weights instead

page = (folder / "index.html").read_text(encoding="utf-8")
data = (folder / "airaware_export.json").read_text(encoding="utf-8")
json.loads(data)  # stop early if the export is not valid JSON

# Hand the export to the page before its own script runs.
inject = "<script>window.AIRAWARE_EXPORT=" + data.replace("</", "<\\/") + ";</script>"
page = page.replace("<body>", "<body>" + inject, 1)

display(HTML(
    '<iframe srcdoc="' + html.escape(page, quote=True) + '" '
    'style="width:100%;height:900px;border:0;border-radius:12px"></iframe>'
))
